In [22]:
import pandas as pd

# 1. Load the dataset using the exact file name
file_name = "Invistico_Airline.csv"
df = pd.read_csv(file_name)

# 2. Inspect the dataset dimensions (rows, columns)
print("--- Dataset Shape ---")
print(f"Total Rows: {df.shape[0]}, Total Columns: {df.shape[1]}")

# 3. Inspect the target variable distribution
print("\n--- Target Variable ('satisfaction') Breakdown ---")
print(df['satisfaction'].value_counts())

# 4. Look at the first 5 rows to confirm everything loaded perfectly
print("\n--- Preview of the Data ---")
df.head()

--- Dataset Shape ---
Total Rows: 129880, Total Columns: 22

--- Target Variable ('satisfaction') Breakdown ---
satisfaction
satisfied       71087
dissatisfied    58793
Name: count, dtype: int64

--- Preview of the Data ---


,satisfaction,Customer Type,Age,Type of Travel,Class,Flight Distance,Seat comfort,Departure/Arrival time convenient,Food and drink,Gate location,...,Online support,Ease of Online booking,On-board service,Leg room service,Baggage handling,Checkin service,Cleanliness,Online boarding,Departure Delay in Minutes,Arrival Delay in Minutes
0,satisfied,Loyal Customer,65,Personal Travel,Eco,265,0,0,0,2,...,2,3,3,0,3,5,3,2,0,0.0
1,satisfied,Loyal Customer,47,Personal Travel,Business,2464,0,0,0,3,...,2,3,4,4,4,2,3,2,310,305.0
2,satisfied,Loyal Customer,15,Personal Travel,Eco,2138,0,0,0,3,...,2,2,3,3,4,4,4,2,0,0.0
3,satisfied,Loyal Customer,60,Personal Travel,Eco,623,0,0,0,3,...,3,1,1,0,1,4,1,3,0,0.0
4,satisfied,Loyal Customer,70,Personal Travel,Eco,354,0,0,0,3,...,4,2,2,0,2,4,2,5,0,0.0


In [23]:
# 1. Convert the target variable 'satisfaction' into 1s and 0s
# 1 = Satisfied, 0 = Dissatisfied
df['satisfaction'] = df['satisfaction'].map({'satisfied': 1, 'dissatisfied': 0})

# 2. Identify the other text-based columns that need encoding
categorical_cols = ['Customer Type', 'Type of Travel', 'Class']

# 3. Apply One-Hot Encoding to convert categories into separate binary columns
# drop_first=True prevents multicollinearity by removing the redundant baseline column
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# 4. Handle any missing data in columns like 'Arrival Delay in Minutes' by filling them with the median
if df_encoded.isnull().sum().sum() > 0:
    df_encoded = df_encoded.fillna(df_encoded.median())

# 5. Verify the new structural format of the dataset
print("--- Encoded Dataset Shape ---")
print(f"New Dimensions: {df_encoded.shape[0]} rows, {df_encoded.shape[1]} columns")
print("\n--- Processed Columns List ---")
print(df_encoded.columns.tolist())

--- Encoded Dataset Shape ---
New Dimensions: 129880 rows, 23 columns

--- Processed Columns List ---
['satisfaction', 'Age', 'Flight Distance', 'Seat comfort', 'Departure/Arrival time convenient', 'Food and drink', 'Gate location', 'Inflight wifi service', 'Inflight entertainment', 'Online support', 'Ease of Online booking', 'On-board service', 'Leg room service', 'Baggage handling', 'Checkin service', 'Cleanliness', 'Online boarding', 'Departure Delay in Minutes', 'Arrival Delay in Minutes', 'Customer Type_disloyal Customer', 'Type of Travel_Personal Travel', 'Class_Eco', 'Class_Eco Plus']


In [24]:
from sklearn.model_selection import train_test_split

# 1. Define 'X' as all predictor features (everything except our target column)
X = df_encoded.drop(columns=['satisfaction'])

# 2. Define 'y' as the target variable we want to predict
y = df_encoded['satisfaction']

# 3. Split the data into training (80%) and testing (20%) sets
# random_state=42 ensures that the shuffle is identical every time you run it
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Verify the distribution of our splits
print("--- Data Split Verification ---")
print(f"Total training samples (X_train): {X_train.shape[0]} rows")
print(f"Total testing samples (X_test):   {X_test.shape[0]} rows")
print(f"Number of input features:         {X_train.shape[1]}")

--- Data Split Verification ---
Total training samples (X_train): 103904 rows
Total testing samples (X_test):   25976 rows
Number of input features:         22


In [25]:
from sklearn.linear_model import LogisticRegression

# 1. Initialize the Logistic Regression model
# max_iter=1000 ensures the algorithm has enough iterations to find the best mathematical fit
model = LogisticRegression(max_iter=1000, random_state=42)

# 2. Fit the model using the training data
print("⏳ Training the Logistic Regression model... Please wait...")
model.fit(X_train, y_train)
print("🎯 Model training complete!")

# 3. Use the trained model to make predictions on our hidden test set
y_pred = model.predict(X_test)

# 4. Also calculate the exact probabilities (useful for business cutoff scores)
y_prob = model.predict_proba(X_test)[:, 1]

print("\n--- Predictions Generated ---")
print(f"Generated {len(y_pred)} satisfaction predictions for the test set evaluation.")

⏳ Training the Logistic Regression model... Please wait...
🎯 Model training complete!

--- Predictions Generated ---
Generated 25976 satisfaction predictions for the test set evaluation.


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [27]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, precision_score, recall_score, classification_report

# ==========================================
# STEP 4: Build and Train the Model
# ==========================================
# Initialize and train our binary classification equation
model = LogisticRegression(max_iter=1000, random_state=42)
print("⏳ Training the model on 103,914 passenger profiles... (Please wait)")
model.fit(X_train, y_train)
print("🎯 Model training complete!")

# Make predictions on our hidden test set
y_pred = model.predict(X_test)

# ==========================================
# STEP 5: Generate Evaluation Metrics
# ==========================================
print("\n================ EVALUATION REPORT ================")

# 1. Generate the Raw Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("--- Confusion Matrix Layout ---")
print(f"True Dissatisfied (TN): {cm[0][0]}   |  False Positive (FP): {cm[0][1]}")
print(f"False Negative (FN):   {cm[1][0]}   |  True Satisfied (TP):  {cm[1][1]}")

# 2. Compute Precision and Recall
precision = precision_score(y_test, y_pred)
# Out of all flagged angry customers, how many were actually angry?
recall = recall_score(y_test, y_pred)

print("\n--- Key Performance Scores ---")
print(f"Precision Score: {precision:.4f} (or {precision*100:.2f}%)")
print(f"Recall Score:    {recall:.4f} (or {recall*100:.2f}%)")

print("\n--- Detailed Classification Summary ---")
print(classification_report(y_test, y_pred, target_names=['Dissatisfied', 'Satisfied']))

⏳ Training the model on 103,914 passenger profiles... (Please wait)
🎯 Model training complete!

================ EVALUATION REPORT ================
--- Confusion Matrix Layout ---
True Dissatisfied (TN): 9145   |  False Positive (FP): 2530
False Negative (FN):   2218   |  True Satisfied (TP):  12083

--- Key Performance Scores ---
Precision Score: 0.8269 (or 82.69%)
Recall Score:    0.8449 (or 84.49%)

--- Detailed Classification Summary ---
              precision    recall  f1-score   support

Dissatisfied       0.80      0.78      0.79     11675
   Satisfied       0.83      0.84      0.84     14301

    accuracy                           0.82     25976
   macro avg       0.82      0.81      0.81     25976
weighted avg       0.82      0.82      0.82     25976



/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [28]:
import numpy as np

# 1. Extract coefficients and match them with feature names
coefficients = model.coef_[0]
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': coefficients,
    'Absolute_Impact': np.abs(coefficients)
})

# 2. Sort by absolute value to see the strongest drivers at the very top
feature_importance = feature_importance.sort_values(by='Absolute_Impact', ascending=False).reset_index(drop=True)

print("================ DRIVERS OF CUSTOMER SATISFACTION ================")
print(feature_importance[['Feature', 'Coefficient']])

================ DRIVERS OF CUSTOMER SATISFACTION ================
                              Feature  Coefficient
0     Customer Type_disloyal Customer    -2.415884
1                           Class_Eco    -1.281630
2                      Class_Eco Plus    -1.072075
3              Inflight entertainment     0.631169
4      Type of Travel_Personal Travel    -0.601178
5              Ease of Online booking     0.329245
6                        Seat comfort     0.318180
7                    On-board service     0.281258
8                      Food and drink    -0.218511
9   Departure/Arrival time convenient    -0.215500
10                    Checkin service     0.207711
11                   Leg room service     0.178587
12              Inflight wifi service    -0.168136
13                    Online boarding     0.088252
14                     Online support     0.045147
15                      Gate location     0.031567
16                   Baggage handling     0.026296
17             